# CSI_6_ARI — CW2: Predictive Maintenance Pipeline
**Student ID:** 4214293 | **RI:** T1-M4-S3  
**Models:** Gradient Boosting Classifier · Multi-Layer Perceptron · Random Forest Classifier

---
## Section 1: Exploratory Data Analysis & Experimental Design

### 1.0 — Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Third-party ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Reproducibility ────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Dataset paths ──────────────────────────────────────────────────
TRAIN_PATH = 'SMI_Train_4214293.csv'
TEST_PATH  = 'SMI_Test_800.csv'
OPS_PATH   = 'SMI_Operational_200.csv'

# ── Business cost matrix ───────────────────────────────────────────
COST_FP = 10    # False positive: unnecessary inspection
COST_FN = 500   # False negative: missed failure (catastrophic)

# ── Global plot style ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
})

print('Imports OK')

### 1.1 — Load & Inspect Datasets

In [ ]:
# Load all three dataset splits from CSV
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
ops_raw   = pd.read_csv(OPS_PATH)

print(f'Training : {train_raw.shape}  |  Test : {test_raw.shape}  |  Operational : {ops_raw.shape}')

# Derive feature list by excluding the label column
FEATURES = [c for c in train_raw.columns if c != 'class']
print(f'Features : {len(FEATURES)}  |  Label : class')

In [ ]:
# Quick structural overview (derived from main.ipynb)
train_raw.info()

In [ ]:
train_raw.head()

### 1.2 — Handle `na` Encoding & Convert to Numeric

In [ ]:
# Count string 'na' tokens before conversion to confirm the encoding
na_count = (train_raw[FEATURES] == 'na').sum().sum()
print(f"String 'na' tokens in training features: {na_count:,}")

In [ ]:
def load_and_clean(df, has_label=True):
    """Replace 'na' strings with NaN, cast all features to numeric, binarise label."""
    feat_cols = [c for c in df.columns if c != 'class']
    X = df[feat_cols].replace('na', np.nan).apply(pd.to_numeric, errors='coerce')
    y = (df['class'] == 'pos').astype(int) if has_label else None
    return X, y

In [ ]:
# Apply cleaning to every split; ops set carries no label
X_train_raw, y_train = load_and_clean(train_raw)
X_test_raw,  y_test  = load_and_clean(test_raw)
X_ops,       _       = load_and_clean(ops_raw, has_label=False)

print(f'X_train : {X_train_raw.shape}  |  X_test : {X_test_raw.shape}  |  X_ops : {X_ops.shape}')
print(f'Positives in training : {y_train.sum()} ({y_train.mean()*100:.2f}%)')

### 1.3 — Class Distribution & Imbalance

In [ ]:
# Absolute counts and percentages for each split
n_neg_tr, n_pos_tr = (y_train == 0).sum(), (y_train == 1).sum()
n_neg_te, n_pos_te = (y_test  == 0).sum(), (y_test  == 1).sum()

print(f'Training — Neg: {n_neg_tr} ({n_neg_tr/len(y_train)*100:.2f}%)  |  Pos: {n_pos_tr} ({n_pos_tr/len(y_train)*100:.2f}%)')
print(f'Test     — Neg: {n_neg_te} ({n_neg_te/len(y_test)*100:.2f}%)  |  Pos: {n_pos_te} ({n_pos_te/len(y_test)*100:.2f}%)')

# Imbalance severity — drives choice of evaluation metrics
print(f'\nImbalance ratio (train) : {n_neg_tr/n_pos_tr:.1f}:1')
print(f'Primary metrics         : Recall, F1(pos), E[Cost] = FP×{COST_FP} + FN×{COST_FN}')

In [ ]:
# Figure 1: Class distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, counts, total, title in [
    (axes[0], [n_neg_tr, n_pos_tr], len(y_train), 'Training Set (n=8,000)'),
    (axes[1], [n_neg_te, n_pos_te], len(y_test),  'Test Set (n=800)')
]:
    bars = ax.bar(['Negative\n(Normal)', 'Positive\n(Failure)'],
                  counts, color=['#2196F3', '#F44336'], edgecolor='white')
    for bar, val in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.005,
                f'{val:,}\n({val/total*100:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title)
    ax.set_ylabel('Count')
    ax.set_ylim(0, max(counts) * 1.18)

fig.suptitle('Figure 1: Class Distribution Across Labelled Datasets', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.4 — Missingness Analysis

In [ ]:
# Overall missingness statistics across all features
miss     = X_train_raw.isnull().sum()
miss_pct = (miss / len(X_train_raw) * 100).round(2)

print(f'Missing cells total    : {miss.sum():,}  ({miss.sum()/X_train_raw.size*100:.2f}% of all cells)')
print(f'Features with any NaN  : {(miss > 0).sum()} / {len(FEATURES)}')
print(f'Features >10% missing  : {(miss_pct > 10).sum()}')
print(f'Features >30% missing  : {(miss_pct > 30).sum()}')
print(f'Max missing            : {miss_pct.max():.2f}%  ({miss_pct.idxmax()})')

In [ ]:
# Class masks for conditional analysis
pos_mask, neg_mask = (y_train == 1), (y_train == 0)

# Per-feature missing rate split by class
miss_pos = X_train_raw[pos_mask].isnull().mean() * 100
miss_neg = X_train_raw[neg_mask].isnull().mean() * 100
overall_miss_pos = miss_pos.mean()
overall_miss_neg = miss_neg.mean()

print(f'Missing rate — Pos: {overall_miss_pos:.2f}%  |  Neg: {overall_miss_neg:.2f}%')
print(f'→ Failures have {overall_miss_pos/overall_miss_neg:.1f}× more missing values (informative missingness)')
print('→ Missingness indicator flags will be added to the preprocessing pipeline')

# Rank features by the magnitude of the pos-neg missingness gap
miss_diff = (miss_pos - miss_neg).sort_values(ascending=False)
print('\nTop 5 features with largest pos-neg gap:')
print(miss_diff.nlargest(5).round(2).to_string())

In [ ]:
# Figure 2: Missing rate per feature
feat_miss = miss_pct[miss_pct > 0].sort_values(ascending=False)
colors = ['#F44336' if v > 30 else '#FF9800' if v > 10 else '#FFC107' for v in feat_miss.values]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(feat_miss)), feat_miss.values, color=colors, width=1.0)
ax.axhline(10, color='#FF9800', linestyle='--', linewidth=1.2, label='10% threshold')
ax.axhline(30, color='#F44336', linestyle='--', linewidth=1.2, label='30% threshold')
ax.legend(handles=[
    mpatches.Patch(color='#F44336', label=f">30% missing  ({(miss_pct>30).sum()} features)"),
    mpatches.Patch(color='#FF9800', label=f"10–30% missing ({((miss_pct>10)&(miss_pct<=30)).sum()} features)"),
    mpatches.Patch(color='#FFC107', label=f"<10% missing  ({((miss_pct>0)&(miss_pct<=10)).sum()} features)"),
], fontsize=9)
ax.set_title('Figure 2: Missing Value Rate per Feature (sorted descending)')
ax.set_xlabel('Features (sorted by missing rate)')
ax.set_ylabel('Missing Rate (%)')
plt.tight_layout()
plt.savefig('fig2_missing_per_feature.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Missing rate by class for top 12 most-discrepant features
top_diff_feats = miss_diff.nlargest(12).index
x, w = np.arange(len(top_diff_feats)), 0.35

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(x - w/2, miss_pos[top_diff_feats].values, w, label='Positive (Failure)', color='#F44336', alpha=0.85)
ax.bar(x + w/2, miss_neg[top_diff_feats].values, w, label='Negative (Normal)',  color='#2196F3', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top_diff_feats, rotation=40, ha='right', fontsize=9)
ax.set_title('Figure 3: Missing Rate by Class — Top 12 Features with Largest Discrepancy')
ax.set_ylabel('Missing Rate (%)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig3_missing_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.5 — Feature Structure: Sparsity, Variance & Skewness

In [ ]:
# Compute zero-value rate, skewness, and variance for every feature
zero_pct = (X_train_raw == 0).sum() / len(X_train_raw) * 100
skew     = X_train_raw.skew()
var      = X_train_raw.var()

# Summarise structural properties
print(f'Sparsity  — >50% zeros : {(zero_pct > 50).sum()}  |  >90% zeros : {(zero_pct > 90).sum()}')
print(f'Skewness  — |skew| >1  : {(skew.abs() > 1).sum()}  |  |skew| >10  : {(skew.abs() > 10).sum()}')
print(f'Variance  — zero-var   : {(var == 0).sum()} → {var[var == 0].index.tolist()}')

In [ ]:
# Figure 4: Sparsity distribution
bins_e   = [0, 25, 50, 75, 90, 100]
labels_b = ['0–25%', '25–50%', '50–75%', '75–90%', '90–100%']
hist_vals = [int(((zero_pct >= bins_e[i]) & (zero_pct < bins_e[i+1])).sum()) for i in range(len(bins_e)-1)]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels_b, hist_vals, color=['#4CAF50','#8BC34A','#FFC107','#FF9800','#F44336'], edgecolor='white')
for bar, val in zip(bars, hist_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Figure 4: Feature Sparsity — Distribution of Zero-Value Rates')
ax.set_xlabel('Zero-Value Rate per Feature')
ax.set_ylabel('Number of Features')
plt.tight_layout()
plt.savefig('fig4_sparsity.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.6 — Class-Conditional Feature Analysis

In [ ]:
# Rank features by absolute difference in class-conditional means
mean_diff = (X_train_raw[pos_mask].mean() - X_train_raw[neg_mask].mean()).abs()
top8      = mean_diff.nlargest(8).index.tolist()

# Tabular summary of top discriminating features
print(f'{"Feature":<15} {"Pos Mean":>12} {"Neg Mean":>12} {"Abs Diff":>12}')
print('─' * 55)
for f in top8:
    pm = X_train_raw[pos_mask][f].mean()
    nm = X_train_raw[neg_mask][f].mean()
    print(f'{f:<15} {pm:>12.1f} {nm:>12.1f} {abs(pm-nm):>12.1f}')

In [ ]:
# Figure 5: Distributions of top 8 discriminating features
fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for i, feat in enumerate(top8):
    ax = axes.flatten()[i]
    pos_vals = X_train_raw[pos_mask][feat].dropna()
    neg_vals = X_train_raw[neg_mask][feat].dropna()
    upper = np.percentile(pd.concat([pos_vals, neg_vals]), 99)
    ax.hist(neg_vals.clip(upper=upper), bins=30, alpha=0.6, color='#2196F3', density=True, label='Neg')
    ax.hist(pos_vals.clip(upper=upper), bins=30, alpha=0.75, color='#F44336', density=True, label='Pos')
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8)

fig.suptitle('Figure 5: Distributions of Top 8 Most Discriminating Features (clipped at 99th pct)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.7 — Correlation & Redundancy Check

In [ ]:
# Spearman is used — robust to the extreme skew identified in §1.5
print('Computing Spearman correlation matrix...')
corr_matrix = X_train_raw.corr(method='spearman')

# Inspect upper triangle only to avoid double-counting pairs
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

print(f'Pairs |corr| > 0.95 : {(upper.abs() > 0.95).sum().sum()}')
print(f'Pairs |corr| > 0.80 : {(upper.abs() > 0.80).sum().sum()}')
print('Note: GBM and RF are robust to correlated features — no removal required.')

In [ ]:
# Figure 6: Correlation heatmap for top 20 discriminating features
top20_feats = mean_diff.nlargest(20).index.tolist()
corr_sub    = corr_matrix.loc[top20_feats, top20_feats]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_sub, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, cbar_kws={'shrink': 0.7})
ax.tick_params(labelsize=8)
ax.set_title('Figure 6: Spearman Correlation — Top 20 Discriminating Features', pad=12)
plt.tight_layout()
plt.savefig('fig6_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.8 — EDA Summary, Risk Register & Experimental Design

In [ ]:
# Collect zero-variance feature names for the summary
zero_var_feats = var[var == 0].index.tolist()

print('=' * 65)
print('EDA SUMMARY — SMI_Train_4214293.csv')
print('=' * 65)

summary = [
    ('Rows',               f'{len(X_train_raw):,}'),
    ('Features',           len(FEATURES)),
    ('Positive (failure)', f'{n_pos_tr} ({n_pos_tr/len(y_train)*100:.2f}%)'),
    ('Imbalance ratio',    f'{n_neg_tr/n_pos_tr:.1f}:1'),
    ('Missing cell rate',  f'{miss.sum()/X_train_raw.size*100:.2f}%'),
    ('Features >30% miss', f'{(miss_pct>30).sum()}'),
    ('Informative miss',   f'Pos={overall_miss_pos:.1f}% vs Neg={overall_miss_neg:.1f}%'),
    ('Zero-variance feat', f'{len(zero_var_feats)} — {zero_var_feats}'),
    ('|Skew| > 10',        f'{(skew.abs()>10).sum()} features'),
    ('Features >90% zero', f'{(zero_pct>90).sum()}'),
]
for k, v in summary:
    print(f'  {k:<25} : {v}')

In [ ]:
print('RISK REGISTER')
print('─' * 65)

risks = [
    ('R1', 'Severe imbalance (59.2:1)',          'SMOTE + class_weight + cost-based threshold'),
    ('R2', 'Informative missingness (pos>>neg)',  'Indicator flags + median imputation (train-fit only)'),
    ('R3', 'Extreme skew in 101 features',        'RobustScaler; log1p transform evaluated'),
    ('R4', 'Zero-variance feature',               'Removed before modelling'),
    ('R5', 'Test positive rate inflation (20%)',  'Accuracy deprioritised; use Recall, F1, E[Cost]'),
]
for rid, risk, mitigation in risks:
    print(f'  {rid}: {risk}')
    print(f'       Mitigation: {mitigation}')

print('=' * 65)

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split

# Stratified 80/20 split — preserves the 1.66% positive rate in both partitions
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_raw, y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

print(f'Training   : {len(X_tr):,}  |  pos={y_tr.sum()} ({y_tr.mean()*100:.2f}%)')
print(f'Validation : {len(X_val):,}  |  pos={y_val.sum()} ({y_val.mean()*100:.2f}%)')
print(f'Test       : {len(X_test_raw):,}  |  pos={y_test.sum()} ({y_test.mean()*100:.2f}%)  [HELD OUT]')

In [ ]:
# 5-fold stratified CV — each fold maintains the minority class ratio
CV_STRATEGY = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f'CV : StratifiedKFold(n_splits=5)  |  ~{y_tr.sum()//5} positives per fold')

In [ ]:
print('Design decisions')
print('─' * 55)

design = [
    ('Imbalance handling',  'SMOTE inside CV folds only (train partition)'),
    ('Leakage prevention',  'All transformers fit on train, applied to val/test'),
    ('Tuning metric',       'Recall (positive class); secondary: F1'),
    ('Threshold selection', 'Minimise E[Cost] = FP×10 + FN×500 on validation'),
]
for k, v in design:
    print(f'  {k:<25} : {v}')

print('\nSection 1 complete — proceed to Section 2: Preprocessing & Model Development.')